# AI Incident Story Draft

This notebook builds a data story from the AI incident dataset step by step.

**Recommended story thesis:** Since 2023, the center of gravity of AI harm has shifted from accidental system failures toward intentional misuse, misinformation, and socially scaled harms.

This notebook is organized to support that claim with a clean sequence of charts.

## 1. Setup

Load the incident, report, and classification tables. The core grain is one row per incident in `incidents.csv`.

In [ ]:
from pathlib import Path
import ast
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

DATA = ROOT / 'data'

incidents = pd.read_csv(DATA / 'incidents.csv')
reports = pd.read_csv(DATA / 'reports.csv')
mit = pd.read_csv(DATA / 'classifications_MIT.csv')
gmf = pd.read_csv(DATA / 'classifications_GMF.csv')
cset = pd.read_csv(DATA / 'classifications_CSETv1.csv')
duplicates = pd.read_csv(DATA / 'duplicates.csv')

incidents['date'] = pd.to_datetime(incidents['date'], errors='coerce')
incidents['year'] = incidents['date'].dt.year

def parse_report_ids(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

incidents['report_ids'] = incidents['reports'].apply(parse_report_ids)
incidents['report_count'] = incidents['report_ids'].apply(len)

print('incidents', incidents.shape)
print('reports', reports.shape)
print('mit', mit.shape)
print('gmf', gmf.shape)
print('cset', cset.shape)

## 2. Quick Structure Check

Use this to confirm the main tables and their roles before charting.

In [ ]:
display(incidents.head(3))
display(mit.head(3))
display(gmf.head(3))
display(cset.head(3))

In [ ]:
summary = pd.DataFrame({
    'table': ['incidents', 'reports', 'mit', 'gmf', 'cset'],
    'rows': [len(incidents), len(reports), len(mit), len(gmf), len(cset)],
    'primary_use': [
        'Main incident-level dataset',
        'Source articles and evidence layer',
        'Risk taxonomy, timing, intent',
        'Technical goals, technology, failure modes',
        'Harm severity, sector, autonomy, rights, public sector'
    ]
})
summary

## 3. Chart 1: Incident Growth Over Time

This establishes scale and shows how sharply incident counts accelerate in recent years.

In [ ]:
year_counts = (
    incidents.dropna(subset=['year'])
    .groupby('year', as_index=False)
    .size()
    .rename(columns={'size': 'incident_count'})
)

ax = sns.lineplot(data=year_counts, x='year', y='incident_count', marker='o', linewidth=3)
ax.set_title('AI Incidents Rise Sharply in Recent Years')
ax.set_xlabel('Year')
ax.set_ylabel('Number of incidents')
plt.tight_layout()
plt.show()

year_counts.tail(10)

## 4. Join MIT Taxonomy to Incident Dates

The MIT table provides the clearest framing for the main story: what type of harm is being reported, when, and whether it was intentional.

In [ ]:
mit_story = mit.merge(
    incidents[['incident_id', 'date', 'year', 'title', 'report_count']],
    left_on='Incident ID',
    right_on='incident_id',
    how='left'
)

mit_story['Risk Domain'] = mit_story['Risk Domain'].fillna('Unknown')
mit_story['Intent'] = mit_story['Intent'].fillna('Unknown')
mit_story[['Incident ID', 'title', 'year', 'Risk Domain', 'Intent']].head()

## 5. Chart 2: Risk Domains by Year

This is the main chart for the story. Focus on 2021 onward so the recent transition is visible.

In [ ]:
recent_mit = mit_story[mit_story['year'] >= 2021].copy()

top_domains = (
    recent_mit['Risk Domain']
    .value_counts()
    .head(5)
    .index
)

risk_by_year = (
    recent_mit[recent_mit['Risk Domain'].isin(top_domains)]
    .groupby(['year', 'Risk Domain'], as_index=False)
    .size()
    .rename(columns={'size': 'incident_count'})
)

ax = sns.lineplot(
    data=risk_by_year,
    x='year',
    y='incident_count',
    hue='Risk Domain',
    marker='o',
    linewidth=3
)
ax.set_title('The Mix of AI Harm Shifts After 2022')
ax.set_xlabel('Year')
ax.set_ylabel('Incident count')
plt.legend(title='Risk Domain', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

risk_by_year.sort_values(['year', 'incident_count'], ascending=[True, False]).head(20)

## 6. Chart 3: Intentional vs Unintentional Incidents

This directly supports the claim that the dominant incident pattern becomes more deliberate over time.

In [ ]:
intent_by_year = (
    recent_mit[recent_mit['Intent'].isin(['Intentional', 'Unintentional'])]
    .groupby(['year', 'Intent'], as_index=False)
    .size()
    .rename(columns={'size': 'incident_count'})
)

ax = sns.barplot(data=intent_by_year, x='year', y='incident_count', hue='Intent')
ax.set_title('Intentional Incidents Overtake Unintentional Ones')
ax.set_xlabel('Year')
ax.set_ylabel('Incident count')
plt.tight_layout()
plt.show()

intent_by_year.pivot(index='year', columns='Intent', values='incident_count').fillna(0)

## 7. Chart 4: Misinformation and Misuse Acceleration

If you want a cleaner presentation chart, isolate the two recent categories that expand fastest.

In [ ]:
focus_domains = [
    '4. Malicious Actors & Misuse',
    '3. Misinformation'
]

focus_risk = (
    recent_mit[recent_mit['Risk Domain'].isin(focus_domains)]
    .groupby(['year', 'Risk Domain'], as_index=False)
    .size()
    .rename(columns={'size': 'incident_count'})
)

ax = sns.lineplot(data=focus_risk, x='year', y='incident_count', hue='Risk Domain', marker='o', linewidth=3)
ax.set_title('Misuse and Misinformation Become Central After 2022')
ax.set_xlabel('Year')
ax.set_ylabel('Incident count')
plt.tight_layout()
plt.show()

## 8. Join GMF Technical Failure Labels

Use the GMF table to explain what is breaking at the technical level behind the visible social harms.

In [ ]:
gmf_story = gmf.merge(
    incidents[['incident_id', 'date', 'year', 'title']],
    left_on='Incident ID',
    right_on='incident_id',
    how='left'
)

gmf_story = gmf_story[gmf_story['Known AI Technical Failure'].notna()].copy()
gmf_story = gmf_story.assign(
    failure=gmf_story['Known AI Technical Failure'].str.split(',')
).explode('failure')
gmf_story['failure'] = gmf_story['failure'].str.strip()
gmf_story = gmf_story[gmf_story['failure'].ne('')]

gmf_story[['Incident ID', 'year', 'title', 'failure']].head()

## 9. Chart 5: Technical Failure Modes Behind Recent Incidents

This gives the audience a technical explanation layer. `Misinformation Generation Hazard` and `Unsafe Exposure or Access` are especially relevant for generative AI era incidents.

In [ ]:
recent_failures = gmf_story[gmf_story['year'] >= 2021].copy()

top_failures = recent_failures['failure'].value_counts().head(6).index
failure_by_year = (
    recent_failures[recent_failures['failure'].isin(top_failures)]
    .groupby(['year', 'failure'], as_index=False)
    .size()
    .rename(columns={'size': 'incident_count'})
)

ax = sns.lineplot(data=failure_by_year, x='year', y='incident_count', hue='failure', marker='o', linewidth=3)
ax.set_title('Recent AI Harms Map to Distinct Failure Modes')
ax.set_xlabel('Year')
ax.set_ylabel('Incident count')
plt.legend(title='Technical failure', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 10. Chart 6: Which Incidents Get the Most Attention?

Because each incident links to many source reports, `report_count` is a reasonable proxy for media attention inside this dataset.

In [ ]:
top_attention = (
    incidents[['incident_id', 'date', 'title', 'report_count']]
    .sort_values('report_count', ascending=False)
    .head(12)
    .sort_values('report_count')
)

ax = sns.barplot(data=top_attention, x='report_count', y='title', palette='crest')
ax.set_title('Some Incidents Dominate Media Attention')
ax.set_xlabel('Linked source reports')
ax.set_ylabel('Incident')
plt.tight_layout()
plt.show()

top_attention[['date', 'title', 'report_count']].sort_values('report_count', ascending=False)

## 11. Optional: Public Sector and Sector-of-Deployment View

Use this if you want a policy-oriented ending rather than a purely technical one.

In [ ]:
cset_clean = cset.copy()
cset_clean['Public Sector Deployment'] = cset_clean['Public Sector Deployment'].fillna('unknown')
cset_clean['Sector of Deployment'] = cset_clean['Sector of Deployment'].fillna('Unknown')

display(cset_clean['Public Sector Deployment'].value_counts(dropna=False))
display(cset_clean['Sector of Deployment'].value_counts().head(12))

## 12. Suggested Final Storyboard

If you need a concise presentation sequence, use this order:

1. Incident growth over time
2. Risk domain shift after 2022
3. Intentional vs unintentional split
4. Technical failure modes behind recent harms
5. Most-covered incidents as concrete case studies

**Narrative close:** AI risk used to be framed mainly as system failure. The recent incident record shows a broader and more urgent reality: AI is increasingly being used intentionally to scale deception, manipulation, and other social harms.